
# 🧪 PyTorch Lab 5 — CNNs (Lab Session 1h30)

> **Format** : séance de TP guidée, manipulation + prédiction + vérification  
> **Durée cible** : 1h30 (bonus en fin si avance rapide)

---

## 🎯 Objectifs
À la fin de ce lab, vous saurez :
1. Mettre des images au format **NCHW** attendu par `Conv2d`
2. Comprendre concrètement l’effet d’un **noyau de convolution**
3. Calculer les **dimensions de sortie** (kernel / stride / padding / dilation)
4. Construire et entraîner un **mini-CNN** de classification
5. (Bonus) Étendre le CNN + visualiser les filtres + appliquer à FashionMNIST


## 0) Setup

In [ ]:

# If you get an ImportError, install dependencies (uncomment if needed):
# !pip -q install torch torchvision matplotlib scikit-learn

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("PyTorch:", torch.__version__)



## 1) Données & format NCHW (digits 8×8)

On utilise `sklearn.datasets.load_digits` (images 8×8, valeurs 0..16).  
Objectif : obtenir un tenseur **(N, C, H, W)** avec **C=1** et valeurs normalisées dans **[0,1]**.



### Exercice 1 — Charger et mettre en forme

**Questions**
1. Quelle est la forme initiale de `X` ?
2. Pourquoi ajouter une dimension de canal ?
3. Quelle forme attend `nn.Conv2d` ?

#### ✍️ À compléter


In [ ]:

digits = load_digits()
X = digits.images.astype(np.float32)  # (N, H, W)
y = digits.target.astype(np.int64)    # (N,)

# 1) Normaliser entre 0 et 1
X = ...

# 2) Convertir en tenseur torch
X_t = ...

# 3) Ajouter dimension canal : (N,H,W) -> (N,1,H,W)
X_t = ...

y_t = torch.from_numpy(y)

print("X (np) shape:", X.shape)
print("X_t (torch) shape:", X_t.shape, "| y_t shape:", y_t.shape)



### Visualisation rapide (sanity check)


In [ ]:

# Visualize a small grid
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for ax, img, label in zip(axes.flatten(), X[:16], y[:16]):
    ax.imshow(img, cmap="gray")
    ax.set_title(int(label))
    ax.axis("off")
plt.tight_layout()
plt.show()



## 2) Convolutions “à la main” avec `F.conv2d` (25 min)

Rappel : `F.conv2d` prend des tenseurs :
- **Input** : (N, C_in, H, W)
- **Kernel/weight** : (C_out, C_in, K_h, K_w)

On va manipuler directement les poids du noyau pour voir l’effet sur l’image.



### Exercice 2 — Noyau identité (3×3)

Construire un noyau 3×3 qui “copie” l’image (type identité : 1 au centre, 0 ailleurs).

#### ✍️ À compléter


In [ ]:

# Batch de 8 images
batch = X_t[:8].clone()

kernel = torch.zeros((1, 1, 3, 3), dtype=batch.dtype)
# Mettre 1 au centre du noyau
kernel[...] = ...

out = F.conv2d(batch, kernel, bias=None, stride=1, padding=0)
print("out shape:", out.shape)



### Visualiser entrée vs sortie

**Question** : pourquoi la taille passe de 8×8 à 6×6 ?



### Exercice 3 — Détection de contours (Sobel / Laplacien)

Tester un filtre **Sobel horizontal** (ou vertical) et observer les zones mises en évidence.


In [ ]:

sobel_h = torch.tensor(
    [[[[-1, 0, 1],
       [-2, 0, 2],
       [-1, 0, 1]]]],
    dtype=torch.float32
)

out_sobel = F.conv2d(batch, sobel_h, stride=1, padding=0)
print("out_sobel shape:", out_sobel.shape)

# Visualisation (entrée vs sobel)
idx = 0
plt.figure(figsize=(7,3))
plt.subplot(1,2,1)
plt.imshow(batch[idx,0], cmap="gray")
plt.title("Input")

plt.subplot(1,2,2)
plt.imshow(out_sobel[idx,0].detach(), cmap="gray")
plt.title("Sobel H")

plt.tight_layout()
plt.show()



> Variante : essayez aussi Laplacien :
\[
\begin{bmatrix}
0 & 1 & 0\\
1 & -4 & 1\\
0 & 1 & 0
\end{bmatrix}
\]


In [ ]:

laplacian = ...

out_lap = F.conv2d(batch, laplacian, stride=1, padding=0)

idx = 0
plt.figure(figsize=(7,3))
plt.subplot(1,2,1)
plt.imshow(batch[idx,0], cmap="gray")
plt.title("Input")

plt.subplot(1,2,2)
plt.imshow(out_lap[idx,0].detach(), cmap="gray")
plt.title("Laplacian")

plt.tight_layout()
plt.show()



## 3) Hyperparamètres & dimensions (20 min)

Formule (sans dilation) :
\[
H_{out} = \left\lfloor \frac{H_{in} + 2P - K}{S} \right\rfloor + 1
\]

Avec dilation \(D\), le **kernel effectif** vaut :
\[
K_{eff} = D\cdot(K-1)+1
\]
et on remplace \(K\) par \(K_{eff}\) dans la formule.



### Exercice 4 — Prédire la taille avant d’exécuter

1) Avec `padding=1` et `stride=2`, que vaut la sortie (H_out, W_out) pour une entrée 8×8 et un kernel 3×3 ?  
2) Vérifier avec PyTorch.



### Exercice 5 — Dilation (optionnel mais instructif)

Tester `dilation=2` avec un kernel 3×3.  
Quel est le kernel effectif \(K_{eff}\) ? Quelle taille de sortie attendue (sans padding, stride=1) ?



## 4) Mini-CNN de classification (30 min)

On va entraîner un petit CNN sur **digits** (10 classes).  
Pipeline :
- Split train/test
- Dataloaders
- Modèle `SmallCNN`
- Boucle d’entraînement courte


In [ ]:

from torch.utils.data import TensorDataset, DataLoader

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_t, y_t, test_size=0.2, random_state=0, stratify=y_t.numpy()
)

train_ds = TensorDataset(X_train, y_train)
test_ds  = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

print("Train:", len(train_ds), "| Test:", len(test_ds))



### Exercice 6 — Compléter l’architecture

Architecture minimale :
- `Conv2d(1 → 8, kernel=3)`
- ReLU
- `MaxPool2d(2)`
- Flatten
- `Linear(...) → 10`

#### ✍️ À compléter


In [ ]:

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = ...
        self.pool = ...
        self.fc = ...
        
    def forward(self, x):
        x = ...
        x = ...
        x = ...
        x = ...
        return x

model = SmallCNN().to(device)
model



### Entraînement

On entraîne rapidement (quelques epochs) et on mesure l’accuracy test.


In [ ]:

def accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.numel()
    return correct / total

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(1, 16):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    test_acc = accuracy(model, test_loader)
    if epoch in [1, 5, 10, 15]:
        print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | test_acc={test_acc*100:.2f}%")



### Afficher quelques prédictions.





# BONUS (si avance rapide)

## Bonus A — Ajouter une 2ᵉ couche de convolution

Objectif : construire un réseau un peu plus expressif :
- Conv(1→8, k=3) + ReLU
- Conv(8→16, k=3) + ReLU
- MaxPool(2)
- FC → 10

> **Astuce** : recalculer la taille avant la FC :
- 8×8 → conv3 → 6×6  
- 6×6 → conv3 → 4×4  
- 4×4 → pool2 → 2×2  
Donc entrée FC = 16×2×2 = 64



## Bonus B — Tester **sans pooling**

Idée : remplacer `MaxPool2d` par un stride dans la convolution (ex : `stride=2`) et comparer.



## Bonus C — Visualiser les filtres appris

On visualise les filtres de la première couche (ici `deep_model.conv1` ou `model.conv`).



## Bonus D — Application à FashionMNIST (plus “réaliste”)

- Images 28×28, 10 classes (vêtements)
- On réutilise l’idée du CNN


---
## ✅ Fin du Lab